# [실습①] 서비스 E2E 테스트 — promptfoo

2일차 2교시. 골든셋으로 서비스를 반복 테스트하고 통과율을 확인합니다.

- **경로 B**: 모델 + 프롬프트 직접 테스트 (`day2/promptfoo/path-b-sample`)
- **경로 A**: 서비스 API 전체 테스트 (`day2/promptfoo/path-a-service`)

In [ ]:
# 실습 저장소 내려받기 + API 키 등록
# API 키는 왼쪽 🔑(보안 비밀) 메뉴에 OPENAI_API_KEY 이름으로 등록해 둡니다. 코드에 직접 붙여 넣지 않습니다.
![ -d /content/ai-quality-lab ] || git clone -q https://github.com/leejaehee1/ai-quality-lab.git /content/ai-quality-lab

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PROMPTFOO_DISABLE_TELEMETRY"] = "1"
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

In [ ]:
# promptfoo 설치 (1~2분)
!npm install -g promptfoo@latest > /dev/null 2>&1
!promptfoo --version

## 경로 B: 공통 예제 챗봇 (프롬프트 v1)

`--repeat 3`: 질문마다 3번 실행 (비결정성 확인) · `--no-cache`: 실제 응답시간 측정

In [ ]:
%cd /content/ai-quality-lab/day2/promptfoo/path-b-sample
!promptfoo eval --repeat 3 --no-cache -o results_b.csv

In [ ]:
import pandas as pd
df = pd.read_csv("results_b.csv")
df.head(30)

### 과제
1. 실패한 테스트를 골라 원인을 추정합니다. 프롬프트, 문서, 모델 중 어디 문제인가요?
2. `prompt_v1.json`을 복사해 `prompt_v2.json`을 만들고 규칙을 보강합니다.
   (왼쪽 파일 탐색기에서 열어 편집할 수 있습니다.)
3. `promptfooconfig.yaml`의 `# - file://prompt_v2.json` 주석을 풀고 다시 실행해 v1과 비교합니다.
4. 어제 만든 골든셋을 `tests`에 추가합니다.

## 경로 A: 서비스 API 전체 테스트

In [ ]:
# 공통 예제 서비스 실행 (같은 런타임 안에서 http://localhost:8000 으로 호출)
# 팀 서비스를 테스트하는 조는 이 셀을 건너뛰고, 아래 SERVICE_URL만 바꿉니다.
!pip install -q fastapi uvicorn openai
!cd /content/ai-quality-lab/sample-service && nohup uvicorn main:app --port 8000 > server.log 2>&1 &

import time, requests
for _ in range(30):
    try:
        print(requests.get("http://localhost:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    print("서버가 뜨지 않았습니다. server.log를 확인하세요:")
    print(open("/content/ai-quality-lab/sample-service/server.log").read())

In [ ]:
%cd /content/ai-quality-lab/day2/promptfoo/path-a-service
!promptfoo eval --repeat 3 --no-cache -o results_a.csv

In [ ]:
pd.read_csv("results_a.csv").head(30)